# Documentation de la classe `WeatherSession`

La classe `WeatherSession` (`kadi.weather.session.WeatherSession`) est la **façade principale** du module Météo de KadiPy. Elle masque toute la complexité des autres classes (gestion du cache, création de l'objet Location, initialisation de la phénologie, etc.) pour offrir à l'utilisateur final une **API unifiée et simplifiée**.

## 1. Initialisation et "Lazy Loading"
Pour démarrer une session, il suffit de fournir les coordonnées de la localité.
L'une des grandes forces de cette façade est son **Lazy Loading** (chargement paresseux) via la méthode privée `_ensure_components()`. Les lourdes requêtes de récupération d'historique ou d'initialisation de l'analyseur hydrologique ne sont effectuées que si l'utilisateur demande spécifiquement un calcul (ex: le bilan hydrique).

In [1]:
# Changer de répertoire
import os
from pathlib import Path
print(Path.cwd())
new_dir = Path("~/Bureau/kadipy/").expanduser().resolve()
os.chdir(new_dir)
print(Path.cwd())

from kadi.weather.session import WeatherSession

# Une simple initialisation instancie la Location et configure le cache, mais ne lance pas d'API lourde.
session = WeatherSession(latitude=9.3333, longitude=2.6333, name='Parakou')
print("Session initialisée avec succès.")

/home/dels/Bureau/kadipy/docs/weather
/home/dels/Bureau/kadipy
Session initialisée avec succès.


## 2. API Unifiée : 9 Fonctions Clés
Une fois la session ouverte, toutes les capacités des sous-modules sont accessibles directement. Voici un aperçu global :

In [ ]:
# --- 1. PRÉVISIONS & HISTORIQUE (Routage vers WeatherData) ---
forecast = session.forecast(days=3)
print("A. Prévision : Source des données = ", forecast['data_source'])

hist_temperatures = session.historical(metric='temperature', months_back=1)
print("B. Historique filtré : lignes récupérées = ", len(hist_temperatures))

# --- 2. PHÉNOLOGIE (Routage vers Phenology) ---
onset = session.onset()
print("\nC. Onset (Début de saison) = ", onset['onset_date'])

gdd = session.growing_degree_days(crop='maize', start_date='2026-05-01', end_date='2026-06-01')
print("D. GDD (Maïs) accumulés = ", gdd['gdd_accumulated'])

cessation = session.cessation()
print("E. Cessation (Fin de saison) estimée = ", cessation['cessation_date'])

# --- 3. HYDROLOGIE (Routage vers Hydrology) ---
et0 = session.et0_hargreaves(tmin=22, tmax=35, day_of_year=150)
print("\nF. ET0 Hargreaves unitaire = ", et0, "mm/j")

wb = session.water_balance(crop='maize', soil_type='ferrugineux')
print("G. Bilan hydrique : jours modélisés = ", len(wb))

# --- 4. RISQUES (Routage vers RiskIndicators) ---
drought = session.drought_index(method='spi', window_months=3)
print("\nH. Sécheresse (SPI) = ", drought['spi_3month'])

rain = session.rain_probability(days_ahead=2)
print("I. Probabilité de pluie recommandée = ", rain['recommendation'])

## 3. Comportement Sous-jacent
Si vous appelez par exemple `session.water_balance()`, le processus suivant s'enclenche :
1. `_ensure_components('hydrology')` vérifie si l'instance `Hydrology` existe.
2. Si elle n'existe pas, `_ensure_data(require_historical=True)` est déclenché pour aller chercher (ou lire en cache) les données CHIRPS et Open-Meteo.
3. L'instance `Hydrology` est créée avec ces données et conservée en RAM.
4. Le calcul du bilan est effectué et retourné.
Cela évite le gaspillage de bande passante et accélère drastiquement les traitements subséquents sur la même session.